# 14.11 Searching and Binary Search

**Prerequisites:** 14.1 Complexity Analysis, 14.10 Sorting, 14.3 Two Pointers  
**Target:** Python 3.12+ (notes flag 3.13/3.14 differences)

### What you'll learn
- Linear vs binary search - and when sorting first is worth it
- 🔴 **The off-by-one minefield**, and one template that avoids all of it
- `bisect` - `bisect_left` vs `bisect_right`, and what each is *for*
- Finding the **first** and **last** occurrence of a duplicated value
- Searching a **rotated** sorted array
- 🔴 **Binary search on the answer** - the pattern behind a whole family of problems
- **Quickselect** - the kth element in O(n) average, without sorting
- Interview questions, worked

---

## The halving

```
   find 23 in [2, 5, 8, 12, 16, 23, 38, 56, 72, 91]

   lo=0                    mid=4                   hi=9
   [2, 5, 8, 12, 16, 23, 38, 56, 72, 91]    16 < 23 -> discard the left half
                     lo=5   mid=7    hi=9
   [.................23, 38, 56, 72, 91]    56 > 23 -> discard the right half
                     lo=5 hi=6
   [.................23, 38...........]     found
```

Each comparison eliminates **half** the remaining candidates. That is the same halving as a balanced BST (**14.7**), and it is why both are O(log n).

| n | Linear (worst) | Binary (worst) |
|---|---|---|
| 1,000 | 1,000 | **10** |
| 1,000,000 | 1,000,000 | **20** |
| 1,000,000,000 | 1,000,000,000 | **30** |

> **A billion items in thirty steps.** Doubling the data costs *one* extra comparison.

### 🔴 The precondition

**The data must be sorted.** Binary search on unsorted data does not error — it returns wrong answers, silently, for some inputs and not others.

| Situation | Do |
|---|---|
| One search, unsorted data | Linear scan, O(n) — sorting costs more |
| Many searches, unsorted | Sort once O(n log n), then O(log n) each |
| Membership only | A `set` — O(1), no sorting (**14.6**) |
| Need neighbours, ranges, or order | **Binary search** — a set cannot do these |

In [ ]:
def linear_search(data, target):
    for index, value in enumerate(data):
        if value == target:
            return index
    return -1


def binary_search(data, target):
    """Classic form. Returns an index, or -1."""
    lo, hi = 0, len(data) - 1              # INCLUSIVE bounds
    steps = 0
    while lo <= hi:                        # <= because hi is inclusive
        steps += 1
        mid = (lo + hi) // 2
        if data[mid] == target:
            return mid, steps
        if data[mid] < target:
            lo = mid + 1                   # 🔴 +1: mid is already ruled out
        else:
            hi = mid - 1                   # 🔴 -1: same reason
    return -1, steps


import math

print(f"{'n':>12}{'binary steps':>15}{'log2(n)':>10}{'linear worst':>15}")
print("-" * 52)
for n in (1_000, 100_000, 1_000_000):
    data = list(range(n))
    _, steps = binary_search(data, n - 1)      # worst case: the last element
    print(f"{n:>12,}{steps:>15}{int(math.log2(n)):>10}{n:>15,}")

print("\n🔴 And on UNSORTED data it fails silently:")
unsorted = [5, 2, 8, 1, 9]
for target in (8, 1):
    index, _ = binary_search(unsorted, target)
    actual = linear_search(unsorted, target)
    print(f"  searching {target} in {unsorted}: binary says {index:>2}, "
          f"truth is {actual}")
print("  ^ no exception. Just a wrong answer for some inputs.")

## 🔴 The off-by-one minefield

Binary search is famously easy to get subtly wrong. Jon Bentley reported that **90% of professional programmers** failed to write a correct version given two hours, and a bug sat in Java's standard library for nine years.

Four independent decisions, each with two plausible options:

```
   1. hi = len(data)   or   len(data) - 1     ?
   2. while lo < hi    or   lo <= hi          ?
   3. lo = mid         or   mid + 1           ?
   4. hi = mid         or   mid - 1           ?
```

Pick inconsistently and you get an infinite loop, an index error, or an answer that is wrong only for empty inputs or the last element.

### The fix: commit to one convention

> Use **half-open** bounds `[lo, hi)` — `lo` is included, `hi` is excluded, exactly like Python slicing and `range()`.

```
    lo, hi = 0, len(data)      hi is one PAST the end
    while lo < hi:             strict <, because hi is excluded
        mid = (lo + hi) // 2
        if <mid is too small>:
            lo = mid + 1       mid ruled out, so skip it
        else:
            hi = mid           mid may be the answer, so KEEP it
    return lo                  lo == hi == the boundary
```

**Why this terminates:** the range shrinks every iteration — `lo` always increases, or `hi` always decreases, because `mid < hi` always holds when `lo < hi`.

> ⚠️ **The overflow note.** In C or Java, `(lo + hi) / 2` can overflow; the fix is `lo + (hi - lo) / 2`. That is the nine-year Java bug. **Python integers are arbitrary precision, so it cannot happen here** — but interviewers ask, so know it.

In [ ]:
def lower_bound(data, target):
    """First index where data[index] >= target. The half-open template.

    Returns len(data) if every element is smaller. This is bisect_left.
    """
    lo, hi = 0, len(data)                  # hi is EXCLUDED
    while lo < hi:                         # strict <
        mid = (lo + hi) // 2
        if data[mid] < target:
            lo = mid + 1                   # mid is too small: rule it out
        else:
            hi = mid                       # mid might be the answer: keep it
    return lo


def upper_bound(data, target):
    """First index where data[index] > target. This is bisect_right.

    ONE character differs from lower_bound: <= instead of <.
    """
    lo, hi = 0, len(data)
    while lo < hi:
        mid = (lo + hi) // 2
        if data[mid] <= target:            # <- the only difference
            lo = mid + 1
        else:
            hi = mid
    return lo


import bisect

data = [1, 3, 3, 3, 5, 8, 8, 10]
print("data:", data)
print(f"index:{'':>6}" + "".join(f"{i:>4}" for i in range(len(data))))
print()
print(f"{'target':>8}{'lower':>8}{'upper':>8}{'bisect_left':>14}{'bisect_right':>14}")
print("-" * 54)
for target in (0, 1, 3, 4, 8, 10, 99):
    print(f"{target:>8}{lower_bound(data, target):>8}{upper_bound(data, target):>8}"
          f"{bisect.bisect_left(data, target):>14}"
          f"{bisect.bisect_right(data, target):>14}")

print("\n  My versions match the standard library exactly.")
print("\n  Read the two columns for target=3: lower=1, upper=4.")
print("  That is the RANGE of 3s: data[1:4]. Their difference, 3, is the")
print("  COUNT of 3s - both in O(log n), with no scanning.")

print("\n  empty input:", lower_bound([], 5), upper_bound([], 5), "- no crash")
print("  single item :", lower_bound([7], 7), upper_bound([7], 7))

### `bisect` - and what people get wrong about it

| Function | Returns | Use for |
|---|---|---|
| `bisect_left(a, x)` | first index where `a[i] >= x` | **the position of x**, or where it would go |
| `bisect_right(a, x)` / `bisect` | first index where `a[i] > x` | **insertion point after** any equals |
| `insort_left/right(a, x)` | — | insert while keeping sorted order |

🔴 **Neither tells you whether `x` is present.** They return an insertion point. To test membership you must check:

```
    i = bisect_left(a, x)
    found = i < len(a) and a[i] == x      <- BOTH conditions
```

🔴 **`insort` is O(n), not O(log n)** (**14.2**). Finding the position is O(log n); actually inserting shifts the array. For repeated insert-and-query, use a heap (**14.8**) or `sortedcontainers`.

### The genuinely useful application: bucketing

Mapping a value to a band — grades, tax brackets, latency histograms — is a `bisect` one-liner, and it is O(log n) rather than a chain of `if`s.

In [ ]:
import bisect


def contains(data, target):
    """The membership test bisect does NOT give you directly."""
    index = bisect.bisect_left(data, target)
    return index < len(data) and data[index] == target


data = [1, 3, 3, 3, 5, 8, 8, 10]
for target in (3, 4, 10, 11):
    index = bisect.bisect_left(data, target)
    print(f"  bisect_left({target:>2}) = {index}  present: {contains(data, target)}")
print("  🔴 index 8 for target 11 is len(data) - indexing it would raise.")


def count_of(data, target):
    """How many times does target appear? O(log n), no scanning."""
    return bisect.bisect_right(data, target) - bisect.bisect_left(data, target)


print(f"\n  count of 3 : {count_of(data, 3)}")
print(f"  count of 8 : {count_of(data, 8)}")
print(f"  count of 4 : {count_of(data, 4)}")


# bucketing - the everyday use
BOUNDARIES = [60, 70, 80, 90]
GRADES = "FDCBA"


def grade(score):
    return GRADES[bisect.bisect_right(BOUNDARIES, score)]


print("\n  score -> grade:")
for score in (33, 59, 60, 71, 89, 90, 100):
    print(f"    {score:>3} -> {grade(score)}")
print("  ^ one line, O(log n), instead of a chain of if/elif")


# and the trap
import time

big = list(range(40_000))
started = time.perf_counter()
for value in range(2_000):
    bisect.insort(big, value)
insort_time = time.perf_counter() - started
print(f"\n  2,000 insorts into a 40,000 list: {insort_time * 1000:.1f} ms")
print("  🔴 The SEARCH is O(log n); the INSERT still shifts the array.")
print("     insort is O(n). Use a heap if you insert constantly (14.8).")

## Searching a rotated sorted array

A sorted array rotated at an unknown point:

```
   original  [0, 1, 2, 4, 5, 6, 7]
   rotated   [4, 5, 6, 7, 0, 1, 2]
```

Not sorted overall — but **at least one half of any split always is**. Find which, then decide whether the target lies inside it.

```
    if data[lo] <= data[mid]:        the LEFT half is sorted
        if data[lo] <= target < data[mid]:  search left
        else:                               search right
    else:                            the RIGHT half is sorted
        if data[mid] < target <= data[hi]:  search right
        else:                               search left
```

Still O(log n). The difficulty is entirely in getting the four comparisons right — and in remembering that a **non-rotated** array is a valid input.

🔴 With **duplicates** the worst case degrades to O(n): `[1, 1, 1, 0, 1]` gives `data[lo] == data[mid] == data[hi]`, and you cannot tell which half is sorted.

In [ ]:
def search_rotated(data, target):
    """O(log n) on a rotated sorted array with distinct values."""
    lo, hi = 0, len(data) - 1
    while lo <= hi:
        mid = (lo + hi) // 2
        if data[mid] == target:
            return mid
        if data[lo] <= data[mid]:              # left half is sorted
            if data[lo] <= target < data[mid]:
                hi = mid - 1
            else:
                lo = mid + 1
        else:                                  # right half is sorted
            if data[mid] < target <= data[hi]:
                lo = mid + 1
            else:
                hi = mid - 1
    return -1


def find_rotation_point(data):
    """Index of the smallest element - where the rotation happened."""
    lo, hi = 0, len(data) - 1
    while lo < hi:
        mid = (lo + hi) // 2
        if data[mid] > data[hi]:
            lo = mid + 1                       # the minimum is to the right
        else:
            hi = mid                           # mid could BE the minimum
    return lo


rotated = [4, 5, 6, 7, 0, 1, 2]
print("array:", rotated)
print("rotation point:", find_rotation_point(rotated),
      "-> smallest is", rotated[find_rotation_point(rotated)])
print()
for target in (0, 4, 2, 3):
    found = search_rotated(rotated, target)
    expected = rotated.index(target) if target in rotated else -1
    print(f"  find {target}: index {found:>2}  correct: {found == expected}")

print("\nexhaustive check over every rotation and every target:")
base = list(range(7))
all_ok = True
for shift in range(len(base)):
    candidate = base[shift:] + base[:shift]
    for target in base + [99]:
        expected = candidate.index(target) if target in candidate else -1
        if search_rotated(candidate, target) != expected:
            all_ok = False
print("  all rotations and targets correct:", all_ok)
print("  ^ including shift=0, which is NOT rotated at all - the case")
print("    that catches implementations assuming a rotation exists.")

---

# 🔴 Binary search on the answer

The most valuable pattern in this notebook, and the least obvious.

So far we searched **an array**. But binary search works on **any range of candidate answers**, provided the answer space is *monotonic*:

> If a candidate `x` works, then everything above (or below) it works too.

```
   candidate answers:   1  2  3  4  5  6  7  8  9  10
   does it work?        N  N  N  N  ✓  ✓  ✓  ✓  ✓  ✓
                                    ^
                        binary search for THIS boundary
```

**How to recognise it.** The question asks for a *minimum* or *maximum* value such that some condition holds, the answer lies in a numeric range, and checking one candidate is much easier than finding the best one.

| Question | Search over |
|---|---|
| Minimum ship capacity to deliver in D days | capacity |
| Slowest eating speed to finish in H hours | speed |
| Smallest largest-sum when splitting into k parts | the sum |
| Maximum minimum distance placing k items | the distance |

**The shape is always the same:**

```
    def works(candidate): ...          O(n) feasibility check

    lo, hi = smallest_possible, largest_possible
    while lo < hi:
        mid = (lo + hi) // 2
        if works(mid): hi = mid        keep it - might be optimal
        else:          lo = mid + 1
    return lo
```

Total cost: **O(n log(range))** instead of trying every candidate.

In [ ]:
def min_ship_capacity(weights, days):
    """Smallest capacity that ships all packages, in order, within `days`.

    O(n log(sum)) - we binary search the CAPACITY, not the array.
    """

    def days_needed(capacity):
        days_used, load = 1, 0
        for weight in weights:
            if load + weight > capacity:
                days_used += 1
                load = 0
            load += weight
        return days_used

    lo = max(weights)              # must fit the heaviest single package
    hi = sum(weights)              # everything in one day
    checks = 0
    while lo < hi:
        checks += 1
        mid = (lo + hi) // 2
        if days_needed(mid) <= days:
            hi = mid               # feasible - try smaller
        else:
            lo = mid + 1           # infeasible - must go bigger
    return lo, checks


weights = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
for days in (1, 5, 10):
    capacity, checks = min_ship_capacity(weights, days)
    print(f"  {days:>2} days -> capacity {capacity:>2}  ({checks} feasibility checks)")

print(f"\n  The candidate range was {max(weights)}..{sum(weights)} = "
      f"{sum(weights) - max(weights) + 1} values.")
print("  Binary search found the boundary in a handful of checks instead")
print("  of trying every one.")

# monotonicity is what makes it valid - let us verify it
target_days = 5
answer, _ = min_ship_capacity(weights, target_days)


def feasible(capacity):
    days_used, load = 1, 0
    for weight in weights:
        if load + weight > capacity:
            days_used, load = days_used + 1, 0
        load += weight
    return days_used <= target_days


pattern = "".join("Y" if feasible(c) else "N"
                  for c in range(max(weights), sum(weights) + 1))
print(f"\n  feasibility across the whole range: {pattern}")
print(f"  answer = {answer}, the first Y at offset {pattern.index('Y')}")
print("\n  🔴 All the Ns come before all the Ys. That MONOTONICITY is the")
print("     precondition - without it, binary search on the answer is wrong.")

## Quickselect - the kth element without sorting

*"What is the kth smallest?"* Sorting gives it in O(n log n). **Quickselect** does it in **O(n) average**.

It is quicksort (**14.10**) that recurses into **only one side**:

```
   partition around a pivot
   if the pivot landed at position k    -> done
   if k < pivot position                -> recurse LEFT only
   else                                 -> recurse RIGHT only
```

**Why that gives O(n):** quicksort does O(n) work per level across log n levels. Quickselect discards one side each time, so the work is n + n/2 + n/4 + … = **2n**.

| | Time | Space |
|---|---|---|
| Sort then index | O(n log n) | O(n) |
| Heap of size k (**14.8**) | O(n log k) | O(k) |
| **Quickselect** | **O(n) average**, 🔴 O(n²) worst | O(1) in place |

🔴 Same worst case as quicksort, same fix: randomise the pivot. (The median-of-medians algorithm guarantees O(n) worst case, but its constants are bad enough that it is rarely used in practice — worth naming, not implementing.)

In [ ]:
import random


def quickselect(data, k, seed=15):
    """kth smallest, 0-indexed. O(n) average with a random pivot."""
    values = list(data)
    rng = random.Random(seed)
    comparisons = 0
    lo, hi = 0, len(values) - 1

    while True:
        if lo == hi:
            return values[lo], comparisons
        pivot = values[rng.randint(lo, hi)]

        smaller, equal, larger = [], [], []
        for value in values[lo:hi + 1]:
            comparisons += 1
            if value < pivot:
                smaller.append(value)
            elif value > pivot:
                larger.append(value)
            else:
                equal.append(value)

        values[lo:hi + 1] = smaller + equal + larger
        pivot_start = lo + len(smaller)
        pivot_end = pivot_start + len(equal) - 1

        if k < pivot_start:
            hi = pivot_start - 1          # recurse LEFT only
        elif k > pivot_end:
            lo = pivot_end + 1            # recurse RIGHT only
        else:
            return values[k], comparisons  # the pivot block contains k


rng = random.Random(15)
sample = [rng.randint(0, 10_000) for _ in range(5_000)]
reference = sorted(sample)

print(f"n = {len(sample):,}\n")
print(f"{'k':>8}{'quickselect':>14}{'sorted[k]':>12}{'comparisons':>14}")
print("-" * 50)
for k in (0, 10, len(sample) // 2, len(sample) - 1):
    value, comparisons = quickselect(sample, k)
    print(f"{k:>8}{value:>14,}{reference[k]:>12,}{comparisons:>14,}")

print(f"\n  n     = {len(sample):,}")
print(f"  2n    = {2 * len(sample):,}   <- the theoretical average")
print(f"  n log n = {int(len(sample) * math.log2(len(sample))):,}   "
      f"<- what sorting would cost")
print("\n  Comparisons land near 2n, not n log n - because each round")
print("  throws away one side entirely.")

all_ok = all(quickselect(sample, k)[0] == reference[k]
             for k in range(0, len(sample), 250))
print(f"\n  agrees with sorted() at 20 values of k: {all_ok}")

## Interview questions

**1. Implement binary search.**
> Use the half-open template. State the precondition (sorted) and mention the `(lo+hi)//2` overflow issue in other languages.

**2. Find the first and last occurrence of a value.**
> `bisect_left` and `bisect_right`. Their difference is the count — both O(log n).

**3. Search in a rotated sorted array.** *(above)*
> Determine which half is sorted, then decide. Handle the non-rotated case, and note duplicates degrade it to O(n).

**4. Find the peak element.**
> Binary search comparing `data[mid]` with `data[mid+1]`; move toward the higher side. O(log n) even though the array is unsorted — the *monotonicity of the slope* is what you search.

**5. Square root of x without `math.sqrt`.**
> Binary search on the answer, over `0..x`. O(log x).

**6. Median of two sorted arrays in O(log(m+n)).**
> Binary search the partition point of the shorter array. Genuinely hard — say so, and offer the O(m+n) merge first.

**7. Kth largest element.** *(above)*
> Quickselect O(n) average, or a size-k heap O(n log k) (**14.8**).

**8. Minimum capacity / Koko eating bananas / split array largest sum.**
> All the same pattern: binary search on the answer with an O(n) feasibility check. Recognising the family is the skill.

**9. When is binary search NOT the right tool?**
> Unsorted data you only search once — sorting costs more than scanning. Membership only — use a `set`. Constantly changing data — insertion is O(n) (**14.2**).

**10. Why is `bisect_left` different from `bisect_right`?**
> With duplicates, left gives the first position and right the position after the last. Their difference is the count of equal elements.

In [ ]:
# Questions 4 and 5 - both are binary search where you would not expect it.
def find_peak(data):
    """An element greater than its neighbours. O(log n) on UNSORTED data.

    We are not searching for a value - we are searching for where the
    slope changes sign, and THAT is monotonic enough to halve.
    """
    lo, hi = 0, len(data) - 1
    while lo < hi:
        mid = (lo + hi) // 2
        if data[mid] < data[mid + 1]:
            lo = mid + 1               # ascending: a peak lies to the right
        else:
            hi = mid                   # descending: mid could be the peak
    return lo


def integer_sqrt(x):
    """Largest integer whose square is <= x. Binary search on the ANSWER."""
    if x < 2:
        return x
    lo, hi = 1, x // 2 + 1
    while lo < hi:
        mid = (lo + hi + 1) // 2       # 🔴 +1 to bias UP, or this loops forever
        if mid * mid <= x:
            lo = mid                   # feasible: keep it
        else:
            hi = mid - 1
    return lo


for values in ([1, 2, 3, 1], [1, 2, 1, 3, 5, 6, 4], [5, 4, 3], [1], [1, 2]):
    peak = find_peak(values)
    left_ok = peak == 0 or values[peak - 1] < values[peak]
    right_ok = peak == len(values) - 1 or values[peak + 1] < values[peak]
    print(f"  peak of {str(values):<24} index {peak} (value {values[peak]})  "
          f"valid: {left_ok and right_ok}")

print()
for x in (0, 1, 4, 8, 15, 16, 26, 1_000_000):
    print(f"  isqrt({x:>9,}) = {integer_sqrt(x):>5}  "
          f"matches math.isqrt: {integer_sqrt(x) == math.isqrt(x)}")

print("\n  🔴 Note `mid = (lo + hi + 1) // 2` in integer_sqrt. When the")
print("     feasible branch does `lo = mid` (not mid+1), a plain floor")
print("     division can leave lo == mid forever. Biasing mid upward is")
print("     the standard fix - and a classic infinite-loop bug.")

---

## Common Mistakes & Pitfalls

1. 🔴 **Binary searching unsorted data.** No error - just wrong answers for some inputs.
2. 🔴 **Mixing the two bound conventions.** Half-open `[lo, hi)` with `while lo < hi`, or inclusive with `while lo <= hi`. Never both.
3. 🔴 **`lo = mid` with plain floor division.** If `mid` can equal `lo`, the loop never advances. Bias `mid` upward with `(lo + hi + 1) // 2`.
4. 🔴 **Treating `bisect_left`'s return as 'found'.** It is an insertion point; check `i < len(a) and a[i] == x`.
5. **Assuming `bisect.insort` is O(log n).** The insert shifts the array - it is O(n) (**14.2**).
6. **Forgetting the non-rotated case** in a rotated-array search.
7. **Applying binary search on the answer without monotonicity.** All the Ns must come before all the Ys.
8. **Using quickselect when you need the top k in order.** It gives you the kth element and a partition, not a sorted top-k.
9. **Sorting to answer one search.** A linear scan is O(n); sorting is O(n log n).

## Best Practices

- Commit to the half-open `[lo, hi)` convention and use it every time.
- Prefer `bisect` to hand-written binary search in production code.
- Use `bisect_right - bisect_left` to count occurrences in O(log n).
- State the sorted precondition out loud before writing the search.
- For 'minimum x such that...', check whether the answer space is monotonic - if it is, binary search it.
- Test empty input, one element, two elements, the first and the last - that is where off-by-ones hide.
- Randomise the pivot in quickselect, as in quicksort (**14.10**).
- Verify a search implementation exhaustively on small inputs against a brute-force version.

## Practice Exercises

Try these before moving on.

1. 🔴 Write binary search from memory, then test it on: empty, one element, two elements, target absent, target first, target last. How many did you get right first time?
2. Implement `bisect_left` recursively and confirm it matches the standard library on 1,000 random arrays.
3. Implement 'Koko eating bananas' - the slowest speed to finish n piles in h hours - and identify the feasibility check and the search range.
4. Extend `search_rotated` to handle duplicates. Why does the worst case become O(n), and which input proves it?
5. Implement 'find the smallest missing positive integer' in a sorted array in O(log n).
6. Compare quickselect, `heapq.nlargest` and `sorted()` for the median of 1,000,000 values. Predict the order before measuring.
7. 🔴 Implement 'median of two sorted arrays' in O(log(min(m,n))). This is genuinely hard - write the O(m+n) merge version first and test against it.
8. Use `bisect` to build a weighted random chooser: given cumulative weights, pick an item in O(log n).